In [1]:
import sys, time, gc, importlib
from pathlib import Path
import duckdb, numpy as np, pandas as pd, lightgbm as lgb
from sklearn.metrics import roc_auc_score

ROOT = Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import src.blocking as B, src.features as F, src.metrics as M

ARTIFACTS = ROOT / "artifacts"
PARQ, KEYS2 = ARTIFACTS / "parquet", ARTIFACTS / "keys_v2"
VALID_GT = ROOT / "notebooks" / "phase2_valid_gt.parquet"
def pq(name): return f"read_parquet('{(PARQ / f'{name}.parquet').as_posix()}')"
def k2(name): return f"read_parquet('{(KEYS2 / f'{name}.parquet').as_posix()}')"

con = duckdb.connect((ARTIFACTS / "blocking.duckdb").as_posix())
B.setup(con, threads=4, memory_limit="4GB", temp_dir=(ARTIFACTS / "duckdb_tmp").as_posix())
F.setup_macros(con)

S1_ALL = k2("train_source1")
TGT = f"(SELECT * FROM {k2('train_source2')} UNION ALL SELECT * FROM {k2('train_source3')})"
print(con.execute("SELECT count(*) FROM cut_full").fetchone())

(11824654,)


In [2]:
F.build_rev(con, "cut_full", "cut_rev")

con.execute(f"""
CREATE OR REPLACE TABLE gt_all AS
SELECT eid(source1_entity_id) AS s1_id, eid(trim(tgt)) AS tgt_id
FROM (SELECT source1_entity_id, unnest(string_split(matched_entity_ids, ',')) AS tgt
      FROM {pq('train_ground_truth')}
      WHERE matched_entity_ids IS NOT NULL AND trim(matched_entity_ids) <> '')
""")

# matcher training S1: not validation, not the reranker's training set
con.execute(f"""
CREATE OR REPLACE TABLE m_tr_s1 AS
SELECT s1_id FROM (
    SELECT DISTINCT s1_id FROM cut_rev
    ANTI JOIN (SELECT eid(source1_entity_id) AS s1_id FROM read_parquet('{VALID_GT.as_posix()}')) v USING (s1_id)
    ANTI JOIN train_s1 USING (s1_id)
) USING SAMPLE 200000 ROWS (reservoir, 21)
""")
con.execute("CREATE OR REPLACE TABLE m_tr_c AS SELECT r.* FROM cut_rev r SEMI JOIN m_tr_s1 USING (s1_id)")
con.execute("CREATE OR REPLACE TABLE m_ev_c AS SELECT r.* FROM cut_rev r SEMI JOIN eval_s1 USING (s1_id)")
print(con.execute("SELECT (SELECT count(*) FROM m_tr_c), (SELECT count(*) FROM m_ev_c)").fetchone())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(1072097, 107242)


In [3]:
def with_labels(cand_tbl, pairs_tbl):
    df = F.build_features(con, cand_tbl, S1_ALL, TGT)
    lab = con.execute(f"""SELECT c.s1_id, c.tgt_id, (g.tgt_id IS NOT NULL)::INT AS y
        FROM {cand_tbl} c LEFT JOIN {pairs_tbl} g USING (s1_id, tgt_id)""").fetchdf()
    return df.merge(lab, on=["s1_id", "tgt_id"])

t0 = time.time()
tr = with_labels("m_tr_c", "gt_all")
ev = with_labels("m_ev_c", "eval_pairs")
print(f"train {tr.shape} pos={tr.y.mean():.3f} | eval {ev.shape}  ({time.time() - t0:.0f}s)")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

train (1072097, 41) pos=0.624 | eval (107242, 41)  (69s)


In [4]:
rng = np.random.default_rng(0)
s1s = tr["s1_id"].unique()
es_ids = set(rng.choice(s1s, size=len(s1s) // 10, replace=False))
es = tr["s1_id"].isin(es_ids)

mt = lgb.LGBMClassifier(n_estimators=3000, learning_rate=0.03, num_leaves=127, min_child_samples=100,
                        subsample=0.8, subsample_freq=1, colsample_bytree=0.8, verbose=-1)
mt.fit(tr.loc[~es, F.FEATS], tr.loc[~es, "y"],
       eval_set=[(tr.loc[es, F.FEATS], tr.loc[es, "y"])],
       callbacks=[lgb.early_stopping(100), lgb.log_evaluation(200)])
mt.booster_.save_model(str(ARTIFACTS / "matcher_lgb.txt"))

ev["q"] = mt.predict_proba(ev[F.FEATS])[:, 1]
print(f"eval AUC reranker p: {roc_auc_score(ev.y, ev.p):.4f} | matcher q: {roc_auc_score(ev.y, ev.q):.4f}")
display(pd.Series(mt.feature_importances_, index=F.FEATS).sort_values(ascending=False).head(20).to_frame("imp").T)

con.register("ev_q", ev[["s1_id", "tgt_id", "q"]])
rows = [M.eval_f05(con, f"(SELECT s1_id, tgt_id FROM ev_q WHERE q >= {t})", f"matcher q>={t}")
        for t in (0.3, 0.4, 0.5, 0.6, 0.7, 0.8)]
display(pd.concat(rows, ignore_index=True))

D:\ML Challenge\6ab10eb3b23ba_student_resource\student_resource\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Training until validation scores don't improve for 100 rounds
[200]	valid_0's binary_logloss: 0.0713662
[400]	valid_0's binary_logloss: 0.0614342
[600]	valid_0's binary_logloss: 0.0593094
[800]	valid_0's binary_logloss: 0.0584586
[1000]	valid_0's binary_logloss: 0.0579759
[1200]	valid_0's binary_logloss: 0.0577022
[1400]	valid_0's binary_logloss: 0.0575883
[1600]	valid_0's binary_logloss: 0.0574758
[1800]	valid_0's binary_logloss: 0.057348
[2000]	valid_0's binary_logloss: 0.0573586
Early stopping, best iteration is:
[1918]	valid_0's binary_logloss: 0.0573071
eval AUC reranker p: 0.9767 | matcher q: 0.9975


,own_sum_p,own_max_p,addr_tset,addr_jacc,t_len,xs_addr_jacc,s_len,p_margin,p_gap_own,xs_name_jw,p,own_n,tgt_max_p,skel_ratio,len_ratio,tsort,cmp_partial,name_jw,tgt_second_p,cmp_jw
imp,17927,15967,12848,12489,12366,12147,10620,10601,9679,9361,8556,8104,8043,7924,7677,6839,6368,5911,5894,5853


,method,macro_f05,f05_singletons,f05_matched,micro_precision,micro_recall,avg_pred
0,matcher q>=0.3,0.956583,0.872778,0.961315,0.969503,0.955727,3.41340
1,matcher q>=0.4,0.960927,0.894294,0.964690,0.976581,0.951424,3.37340
2,matcher q>=0.5,0.964257,0.916745,0.966940,0.982497,0.946760,3.33665
3,matcher q>=0.6,0.966866,0.939196,0.968428,0.987418,0.941749,3.30245
4,matcher q>=0.7,0.967615,0.959775,0.968058,0.990944,0.935439,3.26865
5,matcher q>=0.8,0.967056,0.972872,0.966727,0.993936,0.927728,3.23195


In [5]:
import src.predict as P
importlib.reload(F); importlib.reload(P)
F.setup_macros(con)

matcher = P.load(ARTIFACTS / "matcher_lgb.txt")
del tr; gc.collect()                      # free RAM before the big scoring pass

P.score_all(con, "cut_rev", "q_all", S1_ALL, TGT, matcher, n_chunks=32)

rows = []
for tau in (0.5, 0.6, 0.7, 0.8):
    for o2m in (False, True):
        P.decide(con, "q_all", "pred_tmp", tau, one_to_many=o2m)
        rows.append(M.eval_f05(con, "(SELECT s1_id, tgt_id FROM pred_tmp)",
                               f"q>={tau}{' + one-to-many' if o2m else ''}"))
display(pd.concat(rows, ignore_index=True))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   scored 4/32     117s


...

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   scored 32/32     954s


,method,macro_f05,f05_singletons,f05_matched,micro_precision,micro_recall,avg_pred
0,q>=0.5,0.964257,0.916745,0.966940,0.982497,0.946760,3.33665
1,q>=0.5 + one-to-many,0.965004,0.918616,0.967624,0.983667,0.946182,3.33065
2,q>=0.6,0.966866,0.939196,0.968428,0.987418,0.941749,3.30245
3,q>=0.6 + one-to-many,0.967159,0.940131,0.968685,0.987894,0.941475,3.29990
4,q>=0.7,0.967615,0.959775,0.968058,0.990944,0.935439,3.26865
5,q>=0.7 + one-to-many,0.967625,0.959775,0.968069,0.991004,0.935381,3.26825
6,q>=0.8,0.967056,0.972872,0.966727,0.993936,0.927728,3.23195
7,q>=0.8 + one-to-many,0.967064,0.972872,0.966736,0.993966,0.927699,3.23175


In [6]:
import subprocess, src.rerank as R
importlib.reload(B); importlib.reload(R)

TAU, O2M = 0.7, True                                  # <- set from the Cell 5 table

S1_T = k2("test_source1")
TGT_T_PARTS = [k2("test_source2"), k2("test_source3")]
TGT_T = "(" + " UNION ALL ".join(f"SELECT * FROM {p}" for p in TGT_T_PARTS) + ")"
reranker = R.load(ARTIFACTS / "rerank_lgb.txt")

t0 = time.time()
con.execute("CREATE OR REPLACE TABLE t_cut (s1_id BIGINT, tgt_id BIGINT, src BIGINT, p DOUBLE, rk2 BIGINT)")
B.run_blocking(con, "t_cut", S1_T, TGT_T_PARTS, s1_ids_tbl=None, k_max=30, s1_chunks=32,
               on_chunk=lambda tbl: R.rerank_and_cut(con, tbl, S1_T, TGT_T, reranker, "t_cut", tau=0.01, cap=5))
print(f"blocking done {(time.time() - t0) / 60:.1f} min")

F.build_rev(con, "t_cut", "t_rev")
P.score_all(con, "t_rev", "t_q", S1_T, TGT_T, matcher, n_chunks=32)
P.decide(con, "t_q", "t_pred", TAU, one_to_many=O2M)
print(f"scoring + decision done {(time.time() - t0) / 60:.1f} min")

OUT = ROOT / "output"; OUT.mkdir(exist_ok=True)
cand = P.write_submission(con, "t_cut",  S1_T, TGT_T, "candidate_entity_ids", OUT / "candidate_pairs.tsv")
pred = P.write_submission(con, "t_pred", S1_T, TGT_T, "matched_entity_ids",   OUT / "matching_results.tsv")

n_s1 = con.execute(f"SELECT count(*) FROM {S1_T}").fetchone()[0]
assert len(cand) == len(pred) == n_s1 and pred.source1_entity_id.is_unique
print(f"rows {n_s1:,} | avg candidates {(cand.candidate_entity_ids.str.count(',') + (cand.candidate_entity_ids != '')).mean():.2f}"
      f" | avg matches {(pred.matched_entity_ids.str.count(',') + (pred.matched_entity_ids != '')).mean():.2f}"
      f" | empty predictions {(pred.matched_entity_ids == '').mean():.3f}")

r = subprocess.run([sys.executable, str(ROOT / "utils" / "validate_submission.py"),
                    "--matching", str(OUT / "matching_results.tsv"),
                    "--candidate", str(OUT / "candidate_pairs.tsv"),
                    "--test-dir", str(ROOT / "dataset" / "test")], capture_output=True, text=True)
print(r.stdout, r.stderr)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   t_cut us       token df           8s
   t_cut us       S1 keys           33s
   t_cut us       target keys      125s
   t_cut us       key weights      128s


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

...

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   scored 32/32     984s
scoring + decision done 70.1 min


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

rows 1,732,544 | avg candidates 6.24 | avg matches 3.36 | empty predictions 0.059
ML Challenge 2026 — submission validator
  test dir: D:\ML Challenge\6ab10eb3b23ba_student_resource\student_resource\dataset\test
  required S1 entities: 1732544
  matching_results.tsv: 1732544 rows (102203 empty, 1630341 non-empty).
  candidate_pairs.tsv: 1732544 rows (648 empty, 1731896 non-empty).

PASS — no blocking issues found. Safe to submit.
 


In [7]:
r = subprocess.run([sys.executable, str(ROOT / "utils" / "validate_submission.py"),
                    "--matching", str(OUT / "matching_results.tsv"),
                    "--candidate", str(OUT / "candidate_pairs.tsv"),
                    "--test-dir", str(ROOT / "dataset" / "test"), "--check-ids"],
                   capture_output=True, text=True)
print(r.stdout[-1500:], r.stderr[-1500:])

ML Challenge 2026 — submission validator
  test dir: D:\ML Challenge\6ab10eb3b23ba_student_resource\student_resource\dataset\test
  required S1 entities: 1732544
  valid S2/S3 match IDs: 9969589
  matching_results.tsv: 1732544 rows (102203 empty, 1630341 non-empty).
  candidate_pairs.tsv: 1732544 rows (648 empty, 1731896 non-empty).

PASS — no blocking issues found. Safe to submit.
 


In [8]:
display(con.execute(f"""
SELECT k.country_key,
       count(*)                                  AS n_s1,
       avg(coalesce(c.nc, 0))                    AS avg_candidates,
       avg(coalesce(p.np, 0))                    AS avg_matches,
       avg((coalesce(p.np, 0) = 0)::INT)         AS empty_rate,
       avg(coalesce(b.best_q, 0))                AS avg_best_q,
       avg((b.best_q BETWEEN 0.3 AND 0.7)::INT)  AS uncertain_share,
       avg(coalesce(b.n_hi, 0))                  AS avg_q_above_0_5
FROM (SELECT eid(entity_id) AS s1_id, country_key FROM {S1_T}) k
LEFT JOIN (SELECT s1_id, count(*) AS nc FROM t_cut GROUP BY 1)  c USING (s1_id)
LEFT JOIN (SELECT s1_id, count(*) AS np FROM t_pred GROUP BY 1) p USING (s1_id)
LEFT JOIN (SELECT s1_id, max(q) AS best_q, count(*) FILTER (WHERE q >= 0.5) AS n_hi
           FROM t_q GROUP BY 1) b USING (s1_id)
GROUP BY 1 ORDER BY 1
""").fetchdf())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,country_key,n_s1,avg_candidates,avg_matches,empty_rate,avg_best_q,uncertain_share,avg_q_above_0_5
0,france,259452,7.723987,3.410804,0.053463,0.952344,0.009505,3.499881
1,india,809986,5.966318,3.311100,0.061128,0.943824,0.010836,3.412227
2,us,663106,5.984425,3.390781,0.058541,0.946999,0.009398,3.493043


In [9]:
P.decide(con, "q_all", "pred_best", 0.7, one_to_many=True)

display(con.execute("""
WITH p AS (
    SELECT e.s1_id, e.tgt_id,
           (c2.tgt_id IS NOT NULL) AS in_stage1_top50,
           (cf.tgt_id IS NOT NULL) AS in_cut,
           (pr.tgt_id IS NOT NULL) AS predicted
    FROM eval_pairs e
    LEFT JOIN cand2     c2 USING (s1_id, tgt_id)
    LEFT JOIN cut_full  cf USING (s1_id, tgt_id)
    LEFT JOIN pred_best pr USING (s1_id, tgt_id)
)
SELECT CASE WHEN NOT in_stage1_top50 THEN '1. missed by stage-1 keys'
            WHEN NOT in_cut          THEN '2. dropped by reranker cut'
            WHEN NOT predicted       THEN '3. rejected by matcher / one-to-many'
            ELSE                          '4. found' END AS stage,
       count(*) AS pairs, count(*) / sum(count(*)) OVER () AS share
FROM p GROUP BY 1 ORDER BY 1
""").fetchdf())

print(con.execute("""
SELECT count(*) AS false_positives,
       count(*) / (SELECT count(*) FROM pred_best SEMI JOIN eval_s1 USING (s1_id)) AS fp_share
FROM pred_best pr SEMI JOIN eval_s1 USING (s1_id)
ANTI JOIN eval_pairs USING (s1_id, tgt_id)
""").fetchdf())
display(M.eval_f05(con, "(SELECT s1_id, tgt_id FROM pred_best)", "q>=0.7 + one-to-many"))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,stage,pairs,share
0,1. missed by stage-1 keys,1688,0.024375
1,2. dropped by reranker cut,601,0.008678
2,3. rejected by matcher / one-to-many,2189,0.031609
3,4. found,64774,0.935338


   false_positives  fp_share
0              588  0.008996


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,method,macro_f05,f05_singletons,f05_matched,micro_precision,micro_recall,avg_pred
0,q>=0.7 + one-to-many,0.967625,0.959775,0.968069,0.991004,0.935381,3.26825


In [10]:
fr = con.execute(f"""
WITH s AS (SELECT eid(entity_id) AS s1_id, entity_id, name_v2, name_core, addr_v2
           FROM {S1_T} WHERE country_key = 'france' USING SAMPLE 12 ROWS (reservoir, 3)),
     c AS (SELECT q.*, row_number() OVER (PARTITION BY s1_id ORDER BY q DESC) AS r
           FROM t_q q SEMI JOIN s USING (s1_id))
SELECT s.name_v2 AS s1_name, s.name_core AS s1_core, s.addr_v2 AS s1_addr,
       round(c.q, 3) AS q, t.name_v2 AS tgt_name, t.addr_v2 AS tgt_addr
FROM s
LEFT JOIN c ON c.s1_id = s.s1_id AND c.r <= 4
LEFT JOIN (SELECT eid(entity_id) AS id, name_v2, addr_v2 FROM {TGT_T}) t ON t.id = c.tgt_id
ORDER BY s.s1_id, c.r
""").fetchdf()
pd.set_option("display.max_colwidth", 50)
display(fr)

# most frequent French name / address tokens: reveals unhandled legal forms and abbreviations
display(con.execute(f"""
SELECT 'name' AS f, tok, count(*) AS n
FROM (SELECT unnest(string_split(name_v2, ' ')) AS tok FROM {S1_T} WHERE country_key = 'france')
GROUP BY ALL ORDER BY n DESC LIMIT 30
""").fetchdf().T)
display(con.execute(f"""
SELECT 'addr' AS f, tok, count(*) AS n
FROM (SELECT unnest(string_split(addr_v2, ' ')) AS tok FROM {TGT_T} WHERE country_key = 'france')
GROUP BY ALL ORDER BY n DESC LIMIT 40
""").fetchdf().T)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,s1_name,s1_core,s1_addr,q,tgt_name,tgt_addr
0,maison comice,maison comice,7 rue saint denis nantes pays de la loire,1.000,maison comice,7 rue saint denis nantes loire atlantique
1,maison comice,maison comice,7 rue saint denis nantes pays de la loire,1.000,maison comice,7 rue saint denis nantes loire atlantique
2,maison comice,maison comice,7 rue saint denis nantes pays de la loire,1.000,maison comice and fils,7 rue saint denis nantes loire atlantique
3,maison comice,maison comice,7 rue saint denis nantes pays de la loire,1.000,maison comice et fils,7 rue saint denis nantes pays de la loire
4,union de gitane,union de gitane,14 rue francois joseph nollet lille hauts de f...,1.000,union de gitane,ndeg 14 rue francois joseph nollet lille
5,union de gitane,union de gitane,14 rue francois joseph nollet lille hauts de f...,1.000,union de gitane eurl,ndeg 14 rue francois joseph nollet lille
6,union de gitane,union de gitane,14 rue francois joseph nollet lille hauts de f...,0.998,union de gitane et fils,14 rue francois joseph nollet lille nord
7,union de gitane,union de gitane,14 rue francois joseph nollet lille hauts de f...,0.001,union de gitane france,25 rue francois joseph nollet lille hauts de f...


,0,1,2,3,4,5,6,7,8,9,...,20,21,22,23,24,25,26,27,28,29
f,name,name,name,name,name,name,name,name,name,name,...,name,name,name,name,name,name,name,name,name,name
tok,sarl,sas,club,france,de,eurl,ecole,and,amicale,comite,...,amis,primaire,federation,care,pharmacie,sante,saint,etablissements,la,parents
n,73486,52278,22673,20793,17859,16980,15668,15456,14389,13832,...,6915,6716,6670,6003,5962,5623,5611,5391,5379,5319


,0,1,2,3,4,5,6,7,8,9,...,30,31,32,33,34,35,36,37,38,39
f,addr,addr,addr,addr,addr,addr,addr,addr,addr,addr,...,addr,addr,addr,addr,addr,addr,addr,addr,addr,addr
tok,rue,de,la,loire,bordeaux,nantes,france,hauts,lille,du,...,cap,ferret,ndeg,lege,pornic,bis,l,allee,d,herblain
n,914543,791194,364676,261583,233840,201124,190831,187938,186654,185645,...,51931,50740,46653,43452,42272,41056,40896,40210,39059,37835


In [11]:
def tgt_coverage(cut, pred, tgt_src, label):
    return con.execute(f"""
    SELECT '{label}' AS split, t.country_key, count(*) AS n_targets,
           avg((c.tgt_id IS NOT NULL)::INT) AS in_candidates,
           avg((p.tgt_id IS NOT NULL)::INT) AS in_predictions
    FROM (SELECT eid(entity_id) AS id, country_key FROM {tgt_src}) t
    LEFT JOIN (SELECT DISTINCT tgt_id FROM {cut})  c ON c.tgt_id = t.id
    LEFT JOIN (SELECT DISTINCT tgt_id FROM {pred}) p ON p.tgt_id = t.id
    GROUP BY ALL ORDER BY ALL""").fetchdf()

true_cov = con.execute(f"""
    SELECT t.country_key, avg((g.tgt_id IS NOT NULL)::INT) AS truly_matched
    FROM (SELECT eid(entity_id) AS id, country_key FROM {TGT}) t
    LEFT JOIN (SELECT DISTINCT tgt_id FROM gt_all) g ON g.tgt_id = t.id
    GROUP BY 1 ORDER BY 1""").fetchdf()

display(pd.concat([tgt_coverage("cut_full", "pred_best", TGT, "train"),
                   tgt_coverage("t_cut", "t_pred", TGT_T, "test")], ignore_index=True))
display(true_cov)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,split,country_key,n_targets,in_candidates,in_predictions
0,train,india,4133346,0.908467,0.689444
1,train,us,6186873,0.938847,0.703436
2,test,france,1434993,0.889387,0.616686
3,test,india,4717565,0.877223,0.568502
4,test,us,3817031,0.917070,0.589057


,country_key,truly_matched
0,india,0.740282
1,us,0.740038


In [13]:
def name_core(name_v2):
    """Strip legal forms / honorifics at both ends and trailing country noise:
    'sarl maison comice et fils' -> 'maison comice'."""
    if not name_v2:
        return None
    toks = name_v2.split()
    while len(toks) > 1 and (toks[-1] in LEGAL_FORMS or toks[-1] in TRAILING_NOISE):
        toks.pop()
    while len(toks) > 1 and toks[0] in NAME_PREFIXES:
        toks.pop(0)
    return " ".join(toks)


def normalize_address_v2(value):
    text = to_latin(value)
    if text is None:
        return None
    text = text.replace("g/f", " ground floor ").replace("/", " ")
    text = _canon_tokens(text, ADDR_CANON)
    text = NUM_PREFIX_RE.sub("", text)
    text = NUM_RE.sub(_strip_zeros, text)
    return " ".join(text.split()) or None

In [14]:
importlib.reload(N)
for s in ["Maison Comice et Fils", "Maison Comice & Fils", "SARL Maison Comice", "Union de Gitane France",
          "Ste Generale SA", "Société Générale", "Ets Martin Freres", "Etablissements Martin"]:
    v = N.normalize_name_v2(s); print(f"{s!r:30} -> {v!r:32} core={N.name_core(v)!r}")
for a in ["N° 14 rue François Joseph Nollet, Lille", "14 rue Francois Joseph Nollet Lille", "Plot No 21, Chandan Place"]:
    print(f"{a!r:44} -> {N.normalize_address_v2(a)!r}")

NameError: name 'N' is not defined